In [1]:
# ===============================================================
# 1. Setup and Imports
# ===============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4,5,6,7" # Check using nvidia-smi in terminal and choose GPUs that are not being used
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from haversine import haversine
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


# Detect device
device_type = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using {device_type.upper()} for training")

torch.manual_seed(42)
np.random.seed(42)

✅ Using CUDA for training


In [2]:
# ===============================================================
# 2. Load Dataset
# ===============================================================
CSV_PATH = "osv-5m_subset/train_subset.csv"
IMG_DIR = "osv-5m_subset/train"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")
if not os.path.exists(IMG_DIR):
    raise FileNotFoundError(f"Image folder not found: {IMG_DIR}")

df = pd.read_csv(CSV_PATH)
print(f"📄 Loaded metadata: {df.shape[0]} samples")
display(df.head())

# Verify columns
required = ['id', 'latitude', 'longitude']
for col in required:
    if col not in df.columns:
        raise KeyError(f"Missing required column: {col}")

📄 Loaded metadata: 10000 samples


,id,latitude,longitude,thumb_original_url,country,sequence,captured_at,lon_bin,lat_bin,cell,...,quadtree_10_50000,quadtree_10_12500,quadtree_10_500,quadtree_10_2500,unique_region,unique_sub-region,unique_city,unique_country,creator_username,creator_id
0,149569917138458,45.887541,22.451873,https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...,RO,9l9im94ivovbzy0qwgcpsl,1492440508000,55,74,"(55, 74)",...,205,839,13957,4152,Hunedoara_RO,NaN,Lapugiu de Jos_NaN_Hunedoara_RO,RO,pepenova,1.028911e+14
1,919724982215388,35.245933,140.005689,https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...,JP,eBS_n33kZpKWulqo48738g,1565936860035,88,66,"(88, 66)",...,190,774,12772,3823,Chiba_JP,NaN,Kimitsu_NaN_Chiba_JP,JP,drivephotograph,1.064098e+14
2,729329481758662,-36.143436,147.002683,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,AU,zeJO5Lj16QZCHb8dtrMNgD,1648998548000,90,13,"(90, 13)",...,34,121,1828,605,New South Wales_AU,Albury Municipality_New South Wales_AU,East Albury_Albury Municipality_New South Wale...,AU,skillsy,1.006477e+14
3,1929474523866300,38.465051,-28.288432,https://scontent-cdg4-1.xx.fbcdn.net/m1/v/t6/A...,PT,VAjsepXiS_iq-Gy-d9QAdQ,1526313450615,41,69,"(41, 69)",...,111,456,7542,2305,Azores_PT,Sao Roque do Pico_Azores_PT,Sao Roque do Pico_Sao Roque do Pico_Azores_PT,PT,ligfietser,1.022241e+14
4,205345274735954,57.304645,25.211512,https://scontent-cdg4-3.xx.fbcdn.net/m1/v/t6/A...,LV,ggxzc4t3d92422taut042v,1596196803683,56,83,"(56, 83)",...,243,987,17000,4955,Cesu Rajons_LV,NaN,Cesis_NaN_Cesu Rajons_LV,LV,karlis,1.100973e+14


In [3]:
# ===============================================================
# 3. Dataset Class
# ===============================================================
class OSV5MDataset(Dataset):
    """Dataset for OSV-5M subset with flat directory structure"""
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, f"{row['id']}.jpg")
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        target = torch.tensor([row['latitude'], row['longitude']], dtype=torch.float32)
        return image, target

# Define transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("✅ Dataset class and transforms ready.")

✅ Dataset class and transforms ready.


In [4]:
# ===============================================================
# 4. Lightning DataModule
# ===============================================================
class OSV5MModule(pl.LightningDataModule):
    def __init__(self, df, image_dir, batch_size=8):
        super().__init__()
        self.df = df
        self.image_dir = image_dir
        self.batch_size = batch_size

        # Split train/test 80/20
        n = len(df)
        split = int(0.8 * n)
        self.train_df = df.iloc[:split]
        self.test_df = df.iloc[split:]

        self.transform = transform

    def setup(self, stage=None):
        self.train_dataset = OSV5MDataset(self.train_df, self.image_dir, self.transform)
        self.test_dataset = OSV5MDataset(self.test_df, self.image_dir, self.transform)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

print("✅ DataModule defined.")

✅ DataModule defined.


In [5]:
# ===============================================================
# 5. Lightning Model: ResNet Regressor (with loss_type option)
# ===============================================================
import torch.nn.functional as F

def haversine_loss(pred, target):
    """
    Differentiable Haversine distance loss (in km).
    pred, target: tensors of shape [batch_size, 2] (lat, lon)
    """
    # Convert degrees to radians
    lat1, lon1 = torch.deg2rad(pred[:, 0]), torch.deg2rad(pred[:, 1])
    lat2, lon2 = torch.deg2rad(target[:, 0]), torch.deg2rad(target[:, 1])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = torch.sin(dlat / 2) ** 2 + torch.cos(lat1) * torch.cos(lat2) * torch.sin(dlon / 2) ** 2
    c = 2 * torch.arcsin(torch.sqrt(a))
    km = 6371 * c  # Earth's radius in kilometers
    return km.mean()


class ResNetRegressorPLM(pl.LightningModule):
    def __init__(self, learning_rate=1e-4, backbone="resnet18", loss_type="hybrid", lambda_hav=0.001):
        """
        Args:
            learning_rate (float): optimizer learning rate
            backbone (str): 'resnet18' or 'resnet34'
            loss_type (str): 'mse', 'haversine', or 'hybrid'
            lambda_hav (float): weight for haversine loss when using hybrid
        """
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.loss_type = loss_type
        self.lambda_hav = lambda_hav

        # Choose model backbone
        if backbone == "resnet18":
            self.model = models.resnet18(weights="IMAGENET1K_V1")
        elif backbone == "resnet34":
            self.model = models.resnet34(weights="IMAGENET1K_V1")
        else:
            raise ValueError("Unsupported backbone")

        # Replace final FC layer for regression
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)

        # Define basic MSE for comparison
        self.mse = nn.MSELoss()

    def forward(self, x):
        return self.model(x)

    def compute_loss(self, outputs, targets):
        """Select and compute loss based on loss_type."""
        mse_loss = self.mse(outputs, targets)
        if self.loss_type == "mse":
            return mse_loss
        elif self.loss_type == "haversine":
            return haversine_loss(outputs, targets)
        elif self.loss_type == "hybrid":
            hav_loss = haversine_loss(outputs, targets)
            return mse_loss + self.lambda_hav * hav_loss
        else:
            raise ValueError(f"Unsupported loss type: {self.loss_type}")

    def training_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.compute_loss(outputs, targets)
        rmse = torch.sqrt(self.mse(outputs, targets))
        self.log('train_loss', loss)
        self.log('train_rmse', rmse)
        return loss

    def test_step(self, batch, batch_idx):
        images, targets = batch
        outputs = self(images)
        loss = self.compute_loss(outputs, targets)
        rmse = torch.sqrt(self.mse(outputs, targets))
        self.log('test_loss', loss)
        self.log('test_rmse', rmse)

    def configure_optimizers(self):
        return optim.Adam(self.model.parameters(), lr=self.learning_rate)


print("✅ Model defined (supports MSE, Haversine, and Hybrid).")

✅ Model defined (supports MSE, Haversine, and Hybrid).


In [6]:
# ===============================================================
# 6. Training
# ===============================================================
geo_module = OSV5MModule(df, IMG_DIR, batch_size=8)
geo_model = ResNetRegressorPLM(loss_type="hybrid")

trainer = pl.Trainer(
    max_epochs=10,
    accelerator=device_type,
    devices=1,
    log_every_n_steps=10
)

time_start = time.time()
trainer.fit(geo_model, datamodule=geo_module)
time_stop = time.time()

print(f"⏱ Training completed in {round(time_stop - time_start, 1)} seconds.")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4,5,6,7]

  | Name  | Type    | Params | Mode 
------------------------------------------
0 | model | ResNet  | 11.2 M | train
1 | mse   | MSELoss | 0      | train
------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.710    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode


Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


⏱ Training completed in 284.8 seconds.


In [7]:
# ===============================================================
# 7. Testing and Evaluation (Step 1: better metrics)
# ===============================================================
trainer.test(geo_model, datamodule=geo_module)

geo_model.eval()
preds, trues = [], []

for images, targets in geo_module.test_dataloader():
    with torch.no_grad():
        outputs = geo_model(images)
    preds.extend(outputs.cpu().numpy())
    trues.extend(targets.cpu().numpy())

# ---------- to numpy ----------
preds = np.array(preds)
trues = np.array(trues)

# ---------- clamp to valid lat/lon ranges ----------
preds[:, 0] = np.clip(preds[:, 0], -90, 90)     # latitude
preds[:, 1] = np.clip(preds[:, 1], -180, 180)   # longitude
trues[:, 0] = np.clip(trues[:, 0], -90, 90)
trues[:, 1] = np.clip(trues[:, 1], -180, 180)

# ---------- vectorized haversine (km) ----------
def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    lat1 = np.radians(lat1); lon1 = np.radians(lon1)
    lat2 = np.radians(lat2); lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2*np.arcsin(np.sqrt(a))
    return r*c

errors = haversine_km(trues[:,0], trues[:,1], preds[:,0], preds[:,1])

# ---------- summary that’s robust to outliers ----------
mean_err   = float(np.mean(errors))
median_err = float(np.median(errors))
p90_err    = float(np.percentile(errors, 90))
p95_err    = float(np.percentile(errors, 95))

within_100  = 100.0 * np.mean(errors < 100)
within_500  = 100.0 * np.mean(errors < 500)
within_1000 = 100.0 * np.mean(errors < 1000)

print(f"🌍 Haversine (km) — mean: {mean_err:.2f} | median: {median_err:.2f} | p90: {p90_err:.2f} | p95: {p95_err:.2f}")
print(f"✅ Accuracy — within 100 km: {within_100:.2f}% | within 500 km: {within_500:.2f}% | within 1000 km: {within_1000:.2f}%")

# (optional) inspect worst 5 outliers
# idx = np.argsort(errors)[-5:][::-1]
# for i in idx:
#     print(f"[{i}] err={errors[i]:.0f} km | true=({trues[i,0]:.2f},{trues[i,1]:.2f}) pred=({preds[i,0]:.2f},{preds[i,1]:.2f})")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [4,5,6,7]


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            2042.706787109375
        test_rmse           42.841983795166016
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
🌍 Haversine (km) — mean: 4061.83 | median: 2544.17 | p90: 9428.27 | p95: 12463.56
✅ Accuracy — within 100 km: 0.10% | within 500 km: 6.70% | within 1000 km: 20.40%


In [ ]:
# ===============================================================
# 📘 Summary
# ===============================================================
# - Trained ResNet18 on a 10k OSV-5M subset using transfer learning.
# - Target: latitude and longitude (regression).
# - Output metrics: RMSE (PyTorch Lightning) and Haversine distance (km).
# - Purpose: establish baseline performance for GeoGuessr project.
# - Next steps:
#     • Experiment with ResNet34 or EfficientNet.
#     • Increase dataset to 50k or region-specific splits.
#     • Visualize predicted vs. true locations on map.
# ===============================================================
print("✅ Notebook completed successfully.")

✅ Notebook completed successfully.
